# P5 — Gradient boosting, confidence intervals, and SHAP explanations

**Questions**
1. Does a flexible machine-learning model (**gradient boosting**) beat logistic regression on the *same* 11 predictors?
2. How **uncertain** are our AUROCs? (Confidence intervals, paired comparison)
3. **What** has each model learned? **SHAP** values: for the whole cohort and for one child.

**Gradient boosting in one paragraph:** it builds hundreds of small decision trees one after another. Each new tree corrects the errors of the previous ones. Trees handle **non-linear** effects (a U-shaped risk) and **interactions** (coma matters more if malnourished) automatically, with no z-scores or splines needed. The price: more parameters, which means more risk of overfitting with only 119 deaths, and no simple odds ratios.

⚠️ Synthetic data — methods transfer, exact numbers may not.

## 1. Load data and define both models
- **Logistic**: identical to P3/P4.
- **Boosting** (`HistGradientBoostingClassifier`): handles missing values **natively** (it learns which branch NaN goes to), so it needs no imputer and no scaling.
- Settings are deliberately **conservative** for a small dataset: shallow trees (`max_depth=3`), at least 40 children per leaf, a slow learning rate, and an L2 penalty. With default settings, boosting overfits 119 events easily.

In [ ]:
import itertools, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = pd.read_csv("p2_features.csv")
y = X.pop("inhospital_mortality").to_numpy()
yes = lambda col, val: (X[col] == val).astype(float).where(X[col].notna())
X["caprefill"] = yes("caprefill_adm", "Yes")
X["respdistress"] = yes("respdistress_adm", "Yes")
X["hiv_pos"] = yes("hivstatus_adm", "HIV positive")
X["malaria_pos"] = yes("malariastatuspos_adm", "Yes")
X["not_feeding"] = (X["feedingstatus_adm"] == "Not feeding at all").astype(float)
NUM = ["bcs_total", "waz", "lactate_mmolpl_adm", "spo2_min", "rr_ratio"]
BIN = ["lactate_measured", "caprefill", "respdistress", "hiv_pos", "malaria_pos", "not_feeding"]
PRED = NUM + BIN

def logreg():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM),
        ("bin", SimpleImputer(strategy="most_frequent"), BIN)])
    return Pipeline([("prep", prep), ("lr", LogisticRegression(C=1e6, max_iter=5000))])

def gbm():
    return HistGradientBoostingClassifier(learning_rate=0.03, max_iter=200, max_depth=3,
                                          min_samples_leaf=40, l2_regularization=1.0, random_state=0)

MODELS = {"logistic": logreg, "boosting": gbm}
print(len(PRED), "predictors,", y.sum(), "deaths")

## 2. Head-to-head on the same CV splits
Both models are trained and tested on **exactly the same** 50 folds. That makes the comparison **paired**: we look at the *difference* in each fold, which removes the fold-to-fold noise both models share.

**What to look for:**
- AUROC ≈ 0.745 (boosting) vs 0.753 (logistic). **Boosting is not better.** It wins in fewer than half the folds.
- Brier scores are essentially identical.

This is a common finding: in a systematic review of 71 clinical prediction studies, ML showed **no average benefit** over logistic regression once the comparison was fair (Christodoulou et al., *J Clin Epidemiol* 2019). Tabular clinical data with few events rarely contain enough complex signal for flexibility to pay off.

*(Takes about 15 seconds.)*

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
rows, oof = [], {k: np.zeros(len(y)) for k in MODELS}
for i, (tr, te) in enumerate(cv.split(X, y)):
    for name, make in MODELS.items():
        p = make().fit(X.iloc[tr][PRED], y[tr]).predict_proba(X.iloc[te][PRED])[:, 1]
        rows.append([i, name, roc_auc_score(y[te], p), average_precision_score(y[te], p), brier_score_loss(y[te], p)])
        if i < 5:
            oof[name][te] = p        # first repeat -> one out-of-fold prediction per child

res = pd.DataFrame(rows, columns=["fold", "model", "AUROC", "AUPRC", "Brier"])
diff = res.pivot(index="fold", columns="model", values="AUROC").eval("boosting - logistic")
print(f"Paired AUROC difference (boosting - logistic): {diff.mean():+.3f}; boosting better in {(diff > 0).mean():.0%} of folds")
res.groupby("model")[["AUROC", "AUPRC", "Brier"]].agg(["mean", "std"]).round(3)

## 3. Bootstrap 95% confidence intervals
A single AUROC without a CI hides how much it could change with a different sample. The **bootstrap**:
1. Draw 2,686 children **with replacement** (some appear twice, some not at all).
2. Recompute the AUROC on the out-of-fold predictions.
3. Repeat 1,000 times. The 2.5th and 97.5th percentiles form the 95% CI.

For the **difference**, we resample the *same* children for both models (paired bootstrap).

**What to look for:**
- Each AUROC's CI is about **±0.05** wide (0.70–0.80). That's the price of only 119 deaths.
- The difference CI (about −0.04 to +0.04) **crosses zero** → no evidence either model is better.
- Both are calibrated in the large (mean predicted ≈ 4.4%).

**Reporting rule (TRIPOD+AI):** always give CIs. "AUROC 0.75" and "AUROC 0.75 (0.70–0.80)" tell very different stories.

In [ ]:
rng = np.random.default_rng(0)
boot = []
for _ in range(1000):
    b = rng.integers(0, len(y), len(y))
    boot.append([roc_auc_score(y[b], oof["logistic"][b]), roc_auc_score(y[b], oof["boosting"][b])])
boot = np.array(boot)
ci = lambda a: f"{np.percentile(a, 2.5):.3f} to {np.percentile(a, 97.5):.3f}"
for j, name in enumerate(MODELS):
    print(f"{name:9s} AUROC {roc_auc_score(y, oof[name]):.3f}  (95% CI {ci(boot[:, j])})"
          f"   mean predicted {oof[name].mean():.3f} vs observed {y.mean():.3f}")
d = boot[:, 1] - boot[:, 0]
print(f"difference {d.mean():+.3f}  (95% CI {ci(d)})")

plt.figure(figsize=(7, 3.5))
plt.hist(boot[:, 0], bins=40, alpha=.6, label="logistic"); plt.hist(boot[:, 1], bins=40, alpha=.6, label="boosting")
plt.xlabel("Bootstrap AUROC"); plt.ylabel("Count"); plt.legend(); plt.tight_layout(); plt.show()

## 4. SHAP: what is it?
**The question:** for *this* child, how much did each feature push the predicted risk up or down, relative to an average child?

**Shapley values** come from game theory (Lloyd Shapley, Nobel 2012), where they split a team's payout fairly among players. Here the "players" are features and the "payout" is the prediction:

1. Start from the **base value**: the average prediction over a background sample of children.
2. For a feature *j*, look at every possible **coalition** (subset) of the *other* features. Measure how much the prediction changes when *j* is added to that coalition. "Absent" features take values from the background children.
3. The SHAP value is the **weighted average** of those changes over all coalitions.

**Key property:** base value + sum of SHAP values = the model's prediction for that child, exactly.

We work in **log-odds** (the scale logistic regression uses), so values add up cleanly.

> 💡 In practice you'd use the `shap` library (`shap.TreeExplainer`). Here we compute the values **exactly** ourselves, for teaching. `shap.TreeExplainer` gives the same values for the boosting model (checked to 10⁻⁸), only faster. With 11 features there are only 2¹¹ = 2,048 coalitions. This is precisely what `shap.KernelExplainer` approximates, and seeing it done explicitly makes SHAP much less of a black box.

In [ ]:
def exact_shap(predict, X_explain, X_background):
    p = X_explain.shape[1]
    Xe, Xb = X_explain.to_numpy(float), X_background.to_numpy(float)
    masks = np.array(list(itertools.product([0, 1], repeat=p)), bool)    # all 2^p coalitions
    v = np.zeros((len(Xe), len(masks)))                                   # value of each coalition
    for k, m in enumerate(masks):
        Z = np.repeat(Xb[None], len(Xe), 0)          # start from background children...
        Z[:, :, m] = Xe[:, None, m]                  # ...and plug in this child's values for features in S
        v[:, k] = predict(pd.DataFrame(Z.reshape(-1, p), columns=X_explain.columns)).reshape(len(Xe), -1).mean(1)
    index = {tuple(m): k for k, m in enumerate(masks)}
    phi = np.zeros((len(Xe), p))
    for k, m in enumerate(masks):
        s = m.sum()
        w = math.factorial(s) * math.factorial(p - s - 1) / math.factorial(p) if s < p else 0
        for j in np.where(~m)[0]:
            m2 = m.copy(); m2[j] = True
            phi[:, j] += w * (v[:, index[tuple(m2)]] - v[:, k])          # marginal contribution of j
    return phi, v[:, 0]                                                   # SHAP values, base value

def log_odds(model):
    def f(Z):
        pr = np.clip(model.predict_proba(Z)[:, 1], 1e-9, 1 - 1e-9)
        return np.log(pr / (1 - pr))
    return f
print("defined")

## 5. Compute SHAP for 300 children, both models
We fit both models on **all** data (the models you'd deploy), use 50 random children as the background, and explain 300 random children.

**Sanity checks — what to look for:**
- **Additivity:** base + ΣSHAP equals the model's log-odds (error ≈ 10⁻¹⁵).
- **Logistic:** SHAP = coefficient × (x − background average), exactly. For a linear model SHAP adds nothing new; it re-expresses the coefficients per child. That makes it a good test that our code is right.

*(Takes about 1.5 minutes.)*

In [ ]:
fitted = {name: make().fit(X[PRED], y) for name, make in MODELS.items()}
bg = X[PRED].sample(50, random_state=0)
ex = X[PRED].sample(300, random_state=1)

t = time.time()
shap, base = {}, {}
for name, m in fitted.items():
    shap[name], base[name] = exact_shap(log_odds(m), ex, bg)
print(f"done in {time.time() - t:.0f}s")

for name, m in fitted.items():
    print(f"{name}: additivity error = {np.abs(base[name] + shap[name].sum(1) - log_odds(m)(ex)).max():.1e}")
lr, T = fitted["logistic"], fitted["logistic"]["prep"].transform
linear = (T(ex) - T(bg).mean(0)) * lr["lr"].coef_[0]
print(f"logistic: max |SHAP - coef*(x - mean)| = {np.abs(shap['logistic'] - linear).max():.1e}")

## 6. Global importance: mean |SHAP|
The average size of each feature's push, ignoring direction.

**What to look for:**
- The models **agree** on weight-for-age and respiratory distress as major drivers.
- They **disagree** on breathing rate (`rr_ratio`) and lactate: logistic ≈ 0, boosting gives both high importance. Logistic can only fit a straight line, so if the true shape is curved the line comes out flat. Sections 8 and 9 show this.
- ⚠️ **Blantyre score ranks lower than you'd expect**, yet it had the biggest odds ratio in P3. 91% of children have BCS = 5, so for most children it contributes almost nothing, and the average hides a **rare but huge** effect. Mean |SHAP| measures *typical* impact, not *maximum* impact. The beeswarm plot (§7) shows this.

In [ ]:
imp = pd.DataFrame({name: np.abs(s).mean(0) for name, s in shap.items()}, index=PRED)
imp = imp.sort_values("boosting")
imp.plot.barh(figsize=(7, 5)); plt.xlabel("mean |SHAP| (log-odds)"); plt.tight_layout(); plt.show()
imp.sort_values("boosting", ascending=False).round(3)

## 7. Beeswarm: every child, every feature
Each dot = one child. Horizontal position = SHAP value (right = pushes towards death). Colour = the feature's value (red = high, blue = low, grey = missing).

**What to look for (boosting model):**
- **`bcs_total`:** a dense cluster at ~0 (normal consciousness) plus a **long tail of blue dots far to the right**. That's the comatose children: rare, but their risk rises sharply.
- **`waz`:** blue (low weight-for-age) on the right → malnutrition raises risk; smooth and consistent.
- **`malaria_pos`:** red (positive) on the left → lower risk, same as P1/P3.
- **`lactate_measured`:** all dots at exactly 0. The trees never split on it: once the actual lactate value is known (with NaN handled natively), the "measured" flag is redundant.
- 🚩 **`spo2_min`:** a few **blue dots (low SpO₂) sit far to the left**, so the model says low oxygen saturation *lowers* risk for these children. That's clinically implausible. The likely explanation is **treatment**: hypoxaemic children receive oxygen and many survive, while the trees fit a small group of such survivors. This is the kind of finding SHAP exists to surface: **audit before deploying.**

In [ ]:
def beeswarm(s, X_ex, title):
    order = np.argsort(np.abs(s).mean(0))
    fig, ax = plt.subplots(figsize=(8, 5.5))
    rng = np.random.default_rng(0)
    for row, j in enumerate(order):
        val = X_ex.iloc[:, j]
        col = val.rank(pct=True).to_numpy()
        yy = row + rng.uniform(-.3, .3, len(val))
        miss = val.isna().to_numpy()
        ax.scatter(s[~miss, j], yy[~miss], c=col[~miss], cmap="coolwarm", s=8, vmin=0, vmax=1)
        ax.scatter(s[miss, j], yy[miss], c="grey", s=8)
    ax.axvline(0, c="k", lw=.6)
    ax.set_yticks(range(len(order)), X_ex.columns[order])
    ax.set_xlabel("SHAP value (log-odds)  <- lower risk | higher risk ->"); ax.set_title(title)
    plt.tight_layout(); plt.show()

beeswarm(shap["boosting"], ex, "Boosting — colour: blue = low value, red = high value, grey = missing")

## 8. Dependence plots: the *shape* of each effect
SHAP value against the feature's value.
- **Logistic:** always a straight line by construction.
- **Boosting:** free to bend.

For comparison, the bottom row shows the **raw observed mortality** in six equal-sized groups of the actual data.

**What to look for:**
- **Breathing rate:** raw mortality is **U-shaped**, lowest near `rr_ratio` = 1 (the IMCI threshold) and higher at *both* ends. Very slow breathing in a sick child can mean exhaustion or a depressed brainstem. Boosting captured the U; logistic flattened it to ~0.
- **Lactate:** observed risk rises up to ~2.5 mmol/L and then **plateaus**. Boosting captured the rise, but **above ~5 mmol/L its SHAP falls back below zero**, i.e. "very high lactate is reassuring". That's clinically absurd. Few children have lactate that high, so the trees fit noise in a sparse region. Logistic's gentle straight slope is *wrong in shape* but *safer in the tails*.
- **WAZ:** roughly monotonic, lower = worse. Here the two models agree.

So boosting *did* find real clinical structure (the RR U-shape, the lactate rise), but it also learned noise (the lactate tail), and with 119 deaths the two effects cancel out: its AUROC was not better (§2–3).

In [ ]:
feats = ["rr_ratio", "lactate_mmolpl_adm", "waz"]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for k, f in enumerate(feats):
    j = PRED.index(f)
    for name, mk in [("logistic", "o"), ("boosting", "^")]:
        ax[0, k].scatter(ex[f], shap[name][:, j], s=10, alpha=.6, marker=mk, label=name)
    ax[0, k].axhline(0, c="k", lw=.6); ax[0, k].set(title=f, ylabel="SHAP (log-odds)"); ax[0, k].legend()
    q = pd.qcut(X[f], 6)
    obs = pd.Series(y).groupby(q.values, observed=True).mean() * 100
    mids = X[f].groupby(q.values, observed=True).median()
    ax[1, k].plot(mids, obs, "o-", c="k"); ax[1, k].set(xlabel=f, ylabel="Observed mortality %")
plt.tight_layout(); plt.show()

## 9. Explaining one child (local explanation)
Global plots describe the model. At the bedside, a clinician wants to know **why this child** was flagged.

We take the child with the **highest predicted risk** among the 300 and show each feature's push. This is a "waterfall" in bar form: base value + bars = the child's log-odds.

The "average child" risk is the base value converted from log-odds. It's lower than the 4.4% prevalence because averaging log-odds is not the same as averaging probabilities.

**What to look for:**
- The story is mostly clinically readable: **deep coma** (BCS 1) dominates, then **very slow breathing for age** (`rr_ratio` 0.55, i.e. bradypnoea in a comatose child, an ominous sign), **not feeding**, and **lactate 5.2**.
- 🚩 But **WAZ 0.87 (normal nutrition) pushes risk *up*** a little. That's a small wiggle the trees learned at high WAZ (see §8), and it isn't plausible. A clinician reading this explanation would rightly question it.
- If the top reason were `lactate_measured` or something administrative, that would be a serious warning sign about the model.

In [ ]:
i = int(np.argmax(log_odds(fitted["boosting"])(ex)))
child = ex.iloc[i]
contrib = pd.Series(shap["boosting"][i], index=PRED).sort_values()
risk = 1 / (1 + np.exp(-(base["boosting"][i] + contrib.sum())))
avg = 1 / (1 + np.exp(-base["boosting"][i]))

labels = [f"{f} = {child[f]:.2f}" if pd.notna(child[f]) else f"{f} = missing" for f in contrib.index]
plt.figure(figsize=(8, 5))
plt.barh(labels, contrib, color=np.where(contrib > 0, "tab:red", "tab:blue"))
plt.axvline(0, c="k", lw=.6); plt.xlabel("SHAP (log-odds)")
plt.title(f"Highest-risk child: predicted {risk:.0%} vs average child {avg:.1%}  (died = {y[ex.index[i]]})")
plt.tight_layout(); plt.show()

## 10. What SHAP can and can't tell you
✅ **Can:** show what the *model* relies on, globally and per child; catch models leaning on nonsense or proxies; reveal non-linear shapes.

❌ **Can't:**
- **Causation.** SHAP explains the *model*, not the *disease*. Malaria's negative SHAP doesn't mean malaria protects children. Oxygen use (P1) would get a big positive SHAP, but that reflects clinician concern, not harm.
- **Correlated features.** When two features carry the same information (lactate & lactate_measured, WAZ & MUAC), credit is split between them in ways that depend on the method and the background data.
- **Stability.** A different background sample or a refit on a bootstrap sample can reorder the mid-ranked features. Trust the big, consistent signals only.

## Summary
| | Logistic | Boosting |
|---|---|---|
| CV AUROC | 0.753 | 0.745 |
| 95% CI (out-of-fold) | 0.70–0.80 | 0.69–0.79 |
| Difference | — | −0.004 (−0.04 to +0.04) |
| Captures U-shaped RR / lactate plateau | ✗ | ✓ |
| Interpretability | odds ratios | needs SHAP |

**Key messages**
1. Always run a **fair, paired** comparison against logistic regression. Flexibility doesn't guarantee better performance.
2. Report **confidence intervals**. With 119 events, ±0.05 AUROC is normal.
3. SHAP turns any model into per-child explanations. Use them to **audit** the model, not to infer causes.
4. Non-linearity found by boosting can be brought back into an interpretable model (exercise 1).

## ✍️ Try it yourself
1. **Splines:** give the logistic model a curved RR and lactate effect with `SplineTransformer(n_knots=4)` from `sklearn.preprocessing` in the numeric pipeline (use `C=1.0`). Does CV AUROC change? *(We got 0.753 — same. The curve is real but small.)*
2. **Loosen boosting:** set `max_depth=None, min_samples_leaf=5, max_iter=500`. What happens to train vs CV AUROC? (Hint: P3's kitchen sink.)
3. **Explain a low-risk child:** in §9 replace `argmax` with `argmin`. Which features push risk down?
4. **Stability:** change `bg = X[PRED].sample(50, random_state=0)` to `random_state=7`. Does the importance order in §6 change?

**Next — P6:** synthetic data and privacy, fairness across subgroups (sex, age, HIV, site-like proxies), and whether a Ugandan model would transport to another setting (e.g. your PICU).